# 📏 StandardScaler — Quick Notes
---
> **Dataset:** `tips` (seaborn) | **Library:** `sklearn.preprocessing`

## 📌 Key Points
- **Standardization** → transforms data to have **mean=0, std=1**
- Formula: `z = (x - mean) / std`
- Output is called **Z-score**
- Range is NOT fixed (can be negative, > 1)
- Best for: **Linear models, SVM, KNN, Neural Networks**
- Not affected by outliers as much as MinMax
- **Fit on train only** → transform both train & test

## 🔑 When to Use
| Model | Need Scaling? |
|---|---|
| Linear/Logistic Regression | ✅ Yes |
| SVM / SVR | ✅ Yes (critical!) |
| KNN | ✅ Yes |
| Neural Networks | ✅ Yes |
| Decision Tree / RF | ❌ No |
| XGBoost | ❌ No |

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Load & Encode
df = sns.load_dataset('tips')
df = pd.get_dummies(df, columns=['sex', 'smoker', 'day', 'time'], drop_first=True)
df = df.astype(int)

x = df.drop(columns=['total_bill'])
y = df['total_bill']

# 2. Split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("Before Scaling (x_train):")
print(x_train.describe().round(2))

In [ ]:
# 3. Apply StandardScaler
ss = StandardScaler()

# IMPORTANT: fit on train ONLY — transform both
x_train_scaled = ss.fit_transform(x_train)
x_test_scaled  = ss.transform(x_test)    # ← NO fit_transform here!

df_scaled = pd.DataFrame(x_train_scaled, columns=x_train.columns)
print("After StandardScaler — train set stats:")
print(np.round(df_scaled.describe(), 2))

In [ ]:
# 4. Verify mean ≈ 0, std ≈ 1
print("Mean of each column (should be ≈ 0):")
print(df_scaled.mean().round(4).head())
print("\nStd of each column (should be ≈ 1):")
print(df_scaled.std().round(4).head())

In [ ]:
# 5. Inverse transform — get back original values
x_original = ss.inverse_transform(x_train_scaled)
df_orig = pd.DataFrame(x_original, columns=x_train.columns)
print("Reconstructed original (first 3 rows):")
print(df_orig.head(3).round(2))

## ⚠️ Golden Rule — Fit on Train Only
```python
# ✅ CORRECT
ss = StandardScaler()
x_train = ss.fit_transform(x_train)   # fit + transform
x_test  = ss.transform(x_test)        # transform ONLY

# ❌ WRONG — data leakage!
x_train = ss.fit_transform(x_train)
x_test  = ss.fit_transform(x_test)    # refitting on test = leakage
```

## 🥇 Remember
- After scaling: **mean=0, std=1** for each feature
- Does NOT bound values to a specific range
- `ss.mean_` → mean of each feature from training
- `ss.scale_` → std of each feature from training
- Use `inverse_transform()` to decode back
- Best default scaler for most ML models